# Send low-confidence decisions to a person

Comments come in, and each has to stay up or come down under your community rules. Most are clear; a few are not. Let the model decide the clear ones, and stop for a moderator only when it is unsure: its confidence decides which comments reach a person.

1. Write the workflow: the model's decision, and a moderator's when the model is unsure.
2. Moderate a batch of comments.
3. Let the moderator decide the unsure ones, and finish.

Paste your TypeLLM API key in the cell below, then run all cells.

In [ ]:
%pip install -q "typellm>=0.7.0" pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Paste your API key here.
TYPELLM_API_KEY = ""  # https://typellm.ai/dashboard/keys

In [ ]:
from typellm import TypeLLMClient

client = TypeLLMClient(api_key=TYPELLM_API_KEY)

RULES = ("Community rules: no spam or ads, no harassment or insults aimed at people, no hate speech. "
         "Criticism of ideas is allowed.")

COMMENTS = [
    "Great write-up, the section on caching saved me a day of debugging. Thanks!",
    "Buy followers cheap!!! 10k for $5, DM me on telegram @growfast",
    "You are an idiot and everyone at your company should be fired.",
    "Wow, what a genius take. Truly, nobody has ever thought of this before.",
    "This library is garbage and so is the documentation.",
    "People like the author shouldn't be allowed to write about security.",
    "Check out my blog post on the same topic, I think it adds some context: example.com/caching",
    "lol this is so dumb, I'm dying",
]

## Write the workflow

`decision` returns its probabilities, so it has a confidence. `moderator` is answered by your side, with `"source": "user"`, and only when that confidence is below 0.8. Each path writes its own notice to the author: `notice` after a sure decision, `reviewed_notice` after the moderator's.

In [ ]:
QUESTIONS = {
    "decision": {
        "type": "string",
        "enum": ["allow", "remove"],
        "return_probabilities": True,
        "instructions": "Should this comment stay up under the community rules?",
    },
    "moderator": {
        "type": "string",
        "enum": ["allow", "remove"],
        "source": "user",
        "instructions": "A moderator's decision on this comment.",
        "when": {"decision": {"confidence": {"lt": 0.8}}},
    },
    "notice": {
        "type": "string",
        "depends_on": ["decision"],
        "instructions": "One sentence to the comment's author about the decision.",
        "when": {"decision": {"confidence": {"gte": 0.8}}},
    },
    "reviewed_notice": {
        "type": "string",
        "depends_on": ["moderator"],
        "instructions": "One sentence to the comment's author about the moderator's decision.",
    },
}

## Moderate a batch

Every comment is one call, and the calls run side by side. A sure decision finishes its call; an unsure one stops it with `status` `"needs_input"`.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

import pandas as pd


def moderate(comment):
    return client.generate(context=f"{RULES}\n\nComment: {comment}", questions=QUESTIONS)


with ThreadPoolExecutor(4) as pool:
    responses = list(pool.map(moderate, COMMENTS))

pd.set_option("display.max_colwidth", 70)
pd.DataFrame([{
    "comment": comment,
    "model": response.result["decision"]["value"],
    "confidence": round(response.result["decision"]["confidence"], 2),
    "status": response.status,
} for comment, response in zip(COMMENTS, responses)])

,comment,model,confidence,status
0,"Great write-up, the section on caching saved me a day of debugging. Thanks!",allow,1.00,completed
1,"Buy followers cheap!!! 10k for $5, DM me on telegram @growfast",remove,1.00,completed
2,You are an idiot and everyone at your company should be fired.,remove,1.00,completed
3,"Wow, what a genius take. Truly, nobody has ever thought of this before.",allow,0.73,needs_input
4,This library is garbage and so is the documentation.,allow,0.86,completed
5,People like the author shouldn't be allowed to write about security.,remove,0.41,needs_input
6,"Check out my blog post on the same topic, I think it adds some context: example.com/caching",allow,0.12,needs_input
7,"lol this is so dumb, I'm dying",allow,1.00,completed


## Let the moderator decide the unsure ones

Here the moderator's decisions are a list; in your code, put the comment in a review queue and resume when a person has decided. A stopped call can wait up to 24 hours.

In [ ]:
# What our moderator decided, for the comments that reached them.
MODERATOR = {
    "Wow, what a genius take. Truly, nobody has ever thought of this before.": "allow",
    "People like the author shouldn't be allowed to write about security.": "remove",
    "Check out my blog post on the same topic, I think it adds some context: example.com/caching": "allow",
}

for i, (comment, response) in enumerate(zip(COMMENTS, responses)):
    if response.status == "needs_input":
        responses[i] = client.resume(response, {"moderator": MODERATOR[comment]})

pd.DataFrame([{
    "comment": comment,
    "decided by": "moderator" if "moderator" in response.result else "model",
    "final": response.result.get("moderator", response.result["decision"]["value"]),
    "notice": response.result.get("reviewed_notice") or response.result.get("notice"),
} for comment, response in zip(COMMENTS, responses)])

,comment,decided by,final,notice
0,"Great write-up, the section on caching saved me a day of debugging. Thanks!",model,allow,Your comment has been approved and is now visible to the community.
1,"Buy followers cheap!!! 10k for $5, DM me on telegram @growfast",model,remove,Your comment has been removed for violating the community rule against spam and ads.
2,You are an idiot and everyone at your company should be fired.,model,remove,Your comment was removed for violating the community rule against harassment and insults.
3,"Wow, what a genius take. Truly, nobody has ever thought of this before.",moderator,allow,Your comment has been reviewed and approved for posting.
4,This library is garbage and so is the documentation.,model,allow,Your comment has been allowed as it expresses criticism of the product rather than targeting individuals.
5,People like the author shouldn't be allowed to write about security.,moderator,remove,Your comment was removed because it constitutes harassment by targeting the author personally rather than criticizing their ideas.
6,"Check out my blog post on the same topic, I think it adds some context: example.com/caching",moderator,allow,Your comment has been reviewed and approved for posting.
7,"lol this is so dumb, I'm dying",model,allow,Your comment has been reviewed and is allowed to remain in the community.
